# 05 · PyTorch fundamentals: tensors and gradients

**Objective:** To introduce the core concepts of PyTorch, focusing on its fundamental data structure (the Tensor) and its automatic differentiation engine (`autograd`).

## Part 1: Concepts (90 mins)

### 1. What is PyTorch?

PyTorch is an open-source machine learning library developed by Facebook's AI Research lab. It's widely used for applications such as computer vision and natural language processing. Its two core features are:

- **Tensors:** A multi-dimensional array, similar to NumPy's `ndarray`, but with the ability to run on GPUs for accelerated computing.
- **Automatic Differentiation:** A system called `torch.autograd` that automatically calculates gradients, which are essential for training neural networks.

### 2. Tensors: The Building Blocks
Everything in PyTorch is based on Tensors.

In [2]:
import torch
import numpy as np

# --- Creating Tensors ---

# From a Python list
data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)
print(f"Tensor from list:\n {x_data}\n")

# From a NumPy array (and vice-versa)
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(f"Tensor from NumPy array:\n {x_np}\n")

np_from_tensor = x_np.numpy()
print(f"NumPy array from Tensor:\n {np_from_tensor}\n")

# Tensors of ones, zeros, and random numbers
shape = (2, 3,)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)
rand_tensor = torch.rand(shape)

print(f"Ones Tensor:\n {ones_tensor} \n")
print(f"Random Tensor:\n {rand_tensor} \n")

Tensor from list:
 tensor([[1, 2],
        [3, 4]])

Tensor from NumPy array:
 tensor([[1, 2],
        [3, 4]])

NumPy array from Tensor:
 [[1 2]
 [3 4]]

Ones Tensor:
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Random Tensor:
 tensor([[0.8535, 0.9195, 0.4422],
        [0.3278, 0.3580, 0.7969]]) 



### 3. Tensor Operations
Operations have a syntax similar to NumPy.

In [3]:
tensor = torch.ones(4, 4)

# --- Indexing and Slicing ---
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}\n")

# --- Arithmetic Operations ---
y1 = tensor + tensor
y2 = tensor * tensor

# Matrix multiplication
y3 = tensor.matmul(tensor.T) # Using .matmul()
y4 = tensor @ tensor.T      # Using the @ operator (more common)

print(f"Matrix Multiplication Result:\n {y4}")

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])

Matrix Multiplication Result:
 tensor([[4., 4., 4., 4.],
        [4., 4., 4., 4.],
        [4., 4., 4., 4.],
        [4., 4., 4., 4.]])


### 4. Automatic Differentiation with `torch.autograd`

#### Why do we need derivatives?

**Training a model is an optimisation problem.** A model (from a straight line to a huge neural network) is a function with adjustable numbers, its **parameters** $\theta$ (weights and biases). To measure how good a choice of parameters is, we compute a **loss** $L(\theta)$: a single number that is small when the model's predictions are close to the data (for example the mean squared error). *Training* means finding the parameters that make the loss as small as possible.

**The gradient tells us which way to go.** The gradient $\nabla L = \left(\frac{\partial L}{\partial \theta_1}, \frac{\partial L}{\partial \theta_2}, \dots\right)$ says how much the loss changes when each parameter is nudged a little. It points in the direction in which $L$ *increases* fastest, so a small step in the opposite direction decreases it. This is **gradient descent**:
$$\theta \;\leftarrow\; \theta - \eta \,\nabla L(\theta),$$
repeated many times, where $\eta$ (the *learning rate*) sets the step size. Think of walking downhill in the fog: you cannot see the valley, but you can feel the slope under your feet.

**The problem is scale.** A small neural network has thousands of parameters, a large one billions, and the loss is a long chain of operations (matrix multiplications, activation functions, averages...). We need *all* the partial derivatives, at *every* training step, exactly and fast.

#### How can a computer compute derivatives?

1. **By hand**, then write the formula in code: fine for $3x^2 + 5$, error-prone and practically impossible for a neural network, and it has to be redone every time the model changes.
2. **Numerically**, with finite differences: $\frac{\partial L}{\partial \theta_i} \approx \frac{L(\theta + h\,e_i) - L(\theta)}{h}$. Easy, but (a) only **approximate**: too large an $h$ gives a wrong slope, too small an $h$ drowns in rounding errors; and (b) **slow**: one extra evaluation of the loss *per parameter*, so a million parameters means a million evaluations for a single gradient.
3. **Automatic differentiation (autodiff)**: the approach of PyTorch. Every program, however complicated, is made of elementary operations ($+$, $\times$, $\exp$, $\sin$, matrix product...) whose derivatives are known. While your code runs, PyTorch **records** each operation in a **computation graph**. Then `.backward()` walks the graph **backwards**, from the output to the inputs, combining the local derivatives with the **chain rule**:
$$y = f(g(x)) \quad\Longrightarrow\quad \frac{dy}{dx} = \frac{dy}{dg}\cdot\frac{dg}{dx}.$$
The result is **exact** (up to the usual floating-point precision, no $h$ to choose), and all the partial derivatives with respect to all the inputs come out of **one** backward pass, at a cost comparable to computing the loss itself. This backward walk is what is called *backpropagation* in neural networks.

#### In practice

- We tell PyTorch to track operations on a tensor by setting `requires_grad=True`.
- When we finish our computation (e.g., `y = 3*x**2 + 5`), we call `y.backward()`.
- PyTorch then computes the gradient $dy/dx$ and stores it in the `.grad` attribute of the tensor `x`.

In [4]:
# Create a tensor and set requires_grad=True to track computation
x = torch.tensor(2.0, requires_grad=True)

# Define a simple function
y = 3*x**2 + 5
# In math, the derivative dy/dx is 6*x. At x=2, the gradient should be 12.

# Use autograd to calculate the gradients
y.backward()

# The calculated gradient is now stored in x.grad
print(f"The gradient dy/dx at x=2 is: {x.grad}")

The gradient dy/dx at x=2 is: 12.0


**Looking at the recorded graph.** Every tensor computed from a tensor with `requires_grad=True` remembers the operation that created it in its `.grad_fn` attribute, and `.grad_fn.next_functions` points to the operations before it. For `y = 3*x**2 + 5` the chain is: `x` → power (`x**2`) → multiplication by 3 → addition of 5 → `y`. `backward()` follows these links in the opposite direction, multiplying the local derivatives: $\frac{dy}{dx} = 1 \cdot 3 \cdot 2x = 6x$.

In [5]:
x = torch.tensor(2.0, requires_grad=True)
y = 3*x**2 + 5

print("y:", y)                                   # note grad_fn in the printout
print("Last operation:     ", y.grad_fn)          # the addition of 5
print("Operation before it:", y.grad_fn.next_functions[0][0])   # the multiplication by 3
print("x was created by the user, so x.grad_fn is", x.grad_fn)

y: tensor(17., grad_fn=<AddBackward0>)
Last operation:      <AddBackward0 object at 0x7c236dd70430>
Operation before it: <MulBackward0 object at 0x7c245a447e20>
x was created by the user, so x.grad_fn is None


**Autodiff vs finite differences.** Let's compare the two numerical approaches on $f(x) = \sin(x)\, e^{-x^2/2}$, whose exact derivative at $x = 1$ we can compute by hand: $f'(x) = \left(\cos x - x \sin x\right) e^{-x^2/2}$. PyTorch tensors are `float32` by default (about 7 significant digits), like most deep learning computations.

In [6]:
import math

def f(x):
    return torch.sin(x) * torch.exp(-x**2 / 2)

exact = (math.cos(1) - 1 * math.sin(1)) * math.exp(-1 / 2)
print(f"Exact derivative:   {exact:.8f}\n")

# Finite differences with several step sizes h
x0 = torch.tensor(1.0)
for h in [1e-1, 1e-3, 1e-5, 1e-7]:
    estimate = (f(x0 + h) - f(x0)) / h
    print(f"Finite difference, h = {h:.0e}: {estimate.item():.8f}")

# Automatic differentiation
x0 = torch.tensor(1.0, requires_grad=True)
f(x0).backward()
print(f"\nAutograd:           {x0.grad.item():.8f}")

Exact derivative:   -0.18266804

Finite difference, h = 1e-01: -0.23712397
Finite difference, h = 1e-03: -0.18322466
Finite difference, h = 1e-05: -0.18477440
Finite difference, h = 1e-07: 0.00000000

Autograd:           -0.18266800


Large $h$ gives the slope of a chord, not of the tangent; small $h$ suffers from rounding: with $h = 10^{-7}$, `x0 + h` is *exactly equal* to `x0` in `float32`, and the estimate is 0. There is no good $h$: the best one gets only 2–3 correct digits. Autograd is correct to all the digits `float32` can hold, with no parameter to tune.

**Several inputs, and a trap.** A loss usually depends on many parameters at once. One call to `backward()` fills the `.grad` of *every* tensor with `requires_grad=True` that took part in the computation. But careful: gradients are **accumulated** (added) into `.grad`, not overwritten. If you call `backward()` twice without resetting, you get the sum of the two gradients. This is why training loops reset the gradients at every step.

In [7]:
# A tensor with three parameters, and a loss that depends on all of them
w = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
loss = (w**2).sum()            # L = w1^2 + w2^2 + w3^2, so dL/dw = 2w
loss.backward()
print("Gradient after one backward():", w.grad)

# Same computation again, WITHOUT resetting: the new gradient is ADDED
loss = (w**2).sum()
loss.backward()
print("After a second backward():     ", w.grad, " <- doubled!")

# Reset, then compute again
w.grad.zero_()
loss = (w**2).sum()
loss.backward()
print("After zero_() and backward():  ", w.grad)

Gradient after one backward(): tensor([ 2., -4.,  6.])
After a second backward():      tensor([ 4., -8., 12.])  <- doubled!
After zero_() and backward():   tensor([ 2., -4.,  6.])


### 5. Disabling Gradient Tracking with `torch.no_grad()`

By default, tensors with `requires_grad=True` will track their history. However, there are times when we don't need this, especially during model evaluation (inference). Disabling it saves memory and speeds up computation.

We do this using a `with torch.no_grad():` block.

In [8]:
z = torch.tensor(5.0, requires_grad=True)
print(f"z requires grad: {z.requires_grad}")

with torch.no_grad():
    # Inside this block, operations are not tracked
    q = z**2

print(f"q was created without tracking, so requires_grad is: {q.requires_grad}")

# This would cause an error because the computation history for q was not saved:
try:
    q.backward()
except RuntimeError as e:
    print(f"\nError trying to call backward() on q: {e}")

z requires grad: True
q was created without tracking, so requires_grad is: False

Error trying to call backward() on q: element 0 of tensors does not require grad and does not have a grad_fn


### 6. What autograd is used for: fitting a model with gradient descent

Now we put everything together. We generate noisy data around the line $y = 2x + 1$ and pretend we do not know the slope and the intercept. We start from a wrong guess, $w = 0$ and $b = 0$, and let gradient descent find them. Each step of the loop is exactly what every neural network training does, only with two parameters instead of millions:

1. **Forward pass:** compute the predictions $\hat y = w x + b$ and the loss, the mean squared error $L = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$;
2. **Backward pass:** `loss.backward()` computes $\partial L / \partial w$ and $\partial L / \partial b$;
3. **Update:** move each parameter a small step against its gradient. This is done inside `torch.no_grad()` (section 5): the update itself must not become part of the graph;
4. **Reset** the gradients, so that they do not accumulate into the next step.

We never wrote down a single derivative: autograd did it.

In [9]:
torch.manual_seed(0)

# Data: 100 points around the line y = 2x + 1
x_data = torch.linspace(-1, 1, 100)
y_data = 2 * x_data + 1 + 0.2 * torch.randn(100)

# Parameters to learn, starting from a wrong guess
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

for step in range(101):
    # 1. Forward pass: predictions and loss
    y_pred = w * x_data + b
    loss = ((y_pred - y_data)**2).mean()

    # 2. Backward pass: dL/dw and dL/db
    loss.backward()

    # 3. Update the parameters, without tracking this operation
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    # 4. Reset the gradients for the next step
    w.grad.zero_()
    b.grad.zero_()

    if step % 20 == 0:
        print(f"step {step:3d}: loss = {loss.item():.4f}, w = {w.item():.3f}, b = {b.item():.3f}")

print(f"\nLearned: y = {w.item():.2f} x + {b.item():.2f}   (true line: y = 2 x + 1)")

step   0: loss = 2.4168, w = 0.136, b = 0.201
step  20: loss = 0.1234, w = 1.544, b = 0.998
step  40: loss = 0.0468, w = 1.888, b = 1.007
step  60: loss = 0.0423, w = 1.973, b = 1.007
step  80: loss = 0.0420, w = 1.993, b = 1.007
step 100: loss = 0.0420, w = 1.998, b = 1.007

Learned: y = 2.00 x + 1.01   (true line: y = 2 x + 1)


Try changing `learning_rate`: with `0.01` the parameters crawl towards the solution; with `1.5` the steps are too large and the loss *grows* at every step.

**Where this goes next.** In the next notebooks the same loop comes back with more convenient tools: the model becomes a `torch.nn` module (notebook 06), the update and the reset are done by an *optimizer* from `torch.optim` (`optimizer.step()` and `optimizer.zero_grad()`), and the data is fed in mini-batches (notebooks 07–08). But the core is always: **forward, loss, `backward()`, update.**

**Not only neural networks.** Automatic differentiation is useful whenever you need the derivatives of something you can compute with a program:
* fitting the parameters of any model to data (a physical law, a growth curve, a statistical model) by minimising a loss;
* in physics, forces are minus the gradient of the potential energy, $\mathbf{F} = -\nabla U$: autodiff gives them from the energy function alone;
* sensitivity analysis: how much does the output of a simulation change when an input parameter changes?
* differentiable simulations, optimal control, solving inverse problems...

## Part 2: Exercises & Debugging (90 mins)

### Lab 5.1: Tensor Creation and Manipulation
* **Task:** 
  1. Create a 3x3 tensor with random values.
  2. Create a 3x3 tensor of all ones.
  3. Add the two tensors together.
  4. Multiply the resulting tensor by a scalar (e.g., 5).

In [10]:
# Your code here
shape = (3,3)
random_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)

sum_tensor = random_tensor + ones_tensor
scalar_multiple = sum_tensor * 7

print(f"sum of tensors:\n{sum_tensor}\n")
print(f"scalar multiple:\n{scalar_multiple}\n")


sum of tensors:
tensor([[1.3756, 1.5226, 1.5730],
        [1.6186, 1.6962, 1.5300],
        [1.2560, 1.7366, 1.0204]])

scalar multiple:
tensor([[ 9.6289, 10.6579, 11.0107],
        [11.3301, 11.8735, 10.7097],
        [ 8.7922, 12.1562,  7.1426]])



### Lab 5.2: Simple Linear Equation
* **Task:** Manually perform the calculation for a simple linear layer: `y = Wx + b`.
  1. Create a weight tensor `W` of shape (1, 3) with random values.
  2. Create an input tensor `x` of shape (3, 1) with random values.
  3. Create a bias tensor `b` of shape (1, 1) with a value of 0.5.
  4. Calculate `y` using matrix multiplication (`@`).

In [14]:
# Your code here
w = torch.rand(1,3)
x = torch.rand(3,1)
b = torch.tensor(0.5)  # Example bias term

y = w @ x + b
print(f"result:\n{y}\n")


result:
tensor([[0.6896]])



### Lab 5.3: Practice with Autograd
* **Task:** Calculate the gradient for the function `z = 2a^3 + 3b`.
  1. Create tensors `a` and `b` with values `2.0` and `5.0` respectively. Make sure to set `requires_grad=True`.
  2. Calculate `z`.
  3. Call `backward()` on `z`.
  4. Print the gradients stored in `a.grad` and `b.grad`.
  5. **Question:** Before running, what do you expect the gradients `dz/da` and `dz/db` to be?

In [16]:
# Your code here
a =torch.tensor(2.0, requires_grad=True)
b = torch.tensor(5.0, requires_grad=True)
z = 2*a**3 + 3*b
z.backward()

print(f"The gradient of z with respect to a is: {a.grad}, and with respect to b is: {b.grad}")


The gradient of z with respect to a is: 24.0, and with respect to b is: 3.0


### Debugging Focus: `RuntimeError: element 0 of tensors does not require grad...`
- **The Problem:** You try to call `.backward()` on a tensor, but you get an error saying it doesn't require a gradient.
- **The Cause:** This happens when the tensor you're calling `.backward()` on was not created from other tensors that had `requires_grad=True`. PyTorch didn't build a computation graph, so it doesn't know how to calculate the gradients.
- **The Solution:** Ensure that the input tensors at the very beginning of your calculation have `requires_grad=True` set.